<a href="https://colab.research.google.com/github/garykbrixi/minerva/blob/main/examples/notebooks/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune Minerva with LoRA

Masked-language-model finetuning of Minerva on your own genome, from a GenBank file. LoRA trains
about 1% of the parameters, so it fits on a single Colab GPU.

This uses the same training code as [`scripts/finetune.py`](https://github.com/garykbrixi/minerva/blob/main/scripts/finetune.py):
the `minerva` backbone spec picks the LoRA targets and loss groups, and `MinervaTrainer` applies
Minerva's MLM loss. At the end you can compare the contact map before and after.

Use a GPU runtime (`Runtime` → `Change runtime type`), fill in the form, then `Runtime` → `Run all`.

In [ ]:
import importlib.util
if importlib.util.find_spec("minerva") is None:
    !pip install -q "minerva-dna[viz] @ git+https://github.com/garykbrixi/minerva.git"

In [ ]:
#@title Settings { display-mode: "form" }
data = "twoayggay"  #@param ["twoayggay", "ug27", "upload your own"]
#@markdown `block_size` is the training chunk length in tokens (`--max_seq_length` in `scripts/finetune.py`). Each LOCUS is tiled into blocks and loci are never joined.
block_size    = 4096  #@param {type:"integer"}
#@markdown The default is a short demo run. Raise `max_steps` for a real finetune, and upload a larger genome: the bundled examples are small.
max_steps     = 200   #@param {type:"integer"}
learning_rate = 1e-4  #@param {type:"number"}
lora_r        = 1     #@param {type:"integer"}
lora_alpha    = 2     #@param {type:"integer"}
#@markdown Draw the contact map of the first locus before and after training.
show_before_after = True  #@param {type:"boolean"}
MODEL = "gbrixi/minerva-mlm"  #@param {type:"string"}

## Load the model and data

In [ ]:
import torch
from transformers import AutoTokenizer
from minerva.backbones import get_backbone
from minerva.data import example_path

assert torch.cuda.is_available(), "Finetuning needs a GPU runtime."
# bf16 needs an Ampere or newer GPU. The Colab T4 is older, so it trains in fp16.
bf16 = torch.cuda.get_device_capability()[0] >= 8

if data == "upload your own":
    from google.colab import files   # outside Colab, set gb_file to a path instead
    gb_file = next(iter(files.upload()))
else:
    gb_file = example_path(data)

backbone = get_backbone("minerva")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = backbone.load(MODEL).cuda()

## Contact map before finetuning

In [ ]:
from minerva.data import extract_and_tokenize_gb
from minerva.visualization import publication_head_contacts_rgb

HEADS = ["base_pairing", "repeat", "protein"]

def contact_rgb(model):
    """Head contacts for the first block of the first locus, as an RGB overlay."""
    sequence = extract_and_tokenize_gb(gb_file, use_existing_translations=True)[0]["sequence"]
    tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(sequence))[:block_size]
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16 if bf16 else torch.float16):
        pred = model.eval().predict_contacts(sequence=sequence, tokenizer=tokenizer, head_names=HEADS,
                                             seed_start=0, seed_end=len(tokens), return_dict=True)["predictions"]
    return publication_head_contacts_rgb({h: pred[h].float().cpu().numpy() for h in HEADS}, tokens=tokens)

rgb_before = contact_rgb(model) if show_before_after else None

## Train

Minerva's loss scales each token by the log of its alphabet size, so nucleotide and amino-acid
positions contribute comparably. `backbone.token_groups` supplies those groups.

In [ ]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments
from minerva.finetuning import MinervaTrainer, apply_lora, build_block_dataset

dataset = build_block_dataset(gb_file, tokenizer, block_size=block_size)
print(dataset)

# Checkpoint the encoder directly. Trainer's own hook trips over PEFT wrappers.
backbone.enable_grad_checkpointing(model)
lora_model = apply_lora(model, r=lora_r, alpha=lora_alpha, target_modules=list(backbone.lora_targets))
lora_model.print_trainable_parameters()

trainer = MinervaTrainer(
    model=lora_model,
    token_groups=backbone.token_groups(tokenizer),
    token_type_upweighting=True,
    args=TrainingArguments(
        output_dir="minerva_ft", max_steps=max_steps, learning_rate=learning_rate,
        per_device_train_batch_size=1, gradient_accumulation_steps=2,
        bf16=bf16, fp16=not bf16, logging_steps=10, save_strategy="no",
        report_to=[], remove_unused_columns=False),
    train_dataset=dataset["train"],
    eval_dataset=dataset.get("validation"),
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15),
)
trainer.train()
lora_model.save_pretrained("minerva_lora_adapter")

## Contact map after finetuning

In [ ]:
import matplotlib.pyplot as plt
from minerva.visualization import plot_publication_locus

if show_before_after:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    panels = [(rgb_before, "before"), (contact_rgb(lora_model), f"after {max_steps} steps")]
    for ax, (rgb, title) in zip(axes, panels):
        plot_publication_locus(rgb, title=title, ax=ax, show_legend=ax is axes[1])
    fig.savefig("finetune_before_after.pdf", dpi=600, bbox_inches="tight")
    plt.show()

## Use the adapter

The adapter is saved to `minerva_lora_adapter/`. Load it onto the base model later with:

```python
from peft import PeftModel
from minerva import MinervaForMaskedLM

base = MinervaForMaskedLM.from_pretrained("gbrixi/minerva-mlm")
model = PeftModel.from_pretrained(base, "minerva_lora_adapter")   # .merge_and_unload() for a plain model
```

In [ ]:
import shutil, sys
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(shutil.make_archive("minerva_lora_adapter", "zip", "minerva_lora_adapter"))